# MANIQA Model Forward Walkthrough

이 노트북은 `models/maniqa.py`와 같은 위치에 있으며, MANIQA 모델 내부 `forward()`를 논문 Figure 2 순서대로 읽기 위한 해설 노트북입니다.

실제 실행은 repo root에서 kernel을 잡는 것이 편하지만, 이 파일은 모델 코드 옆에 두어 `maniqa.py`를 보면서 같이 읽을 수 있게 했습니다.

## 1. 코드 위치 매칭

| 논문 Figure 2 단계 | 코드 위치 |
| --- | --- |
| 입력 이미지 | `MANIQA.forward(x)`의 인자 `x` |
| 224x224 crop 또는 resize | `debug_forward.py`의 `load_image_tensor()` |
| patch embedding | `self.vit.patch_embed(x)` |
| ViT feature extraction | `self.vit(x)`와 ViT block forward hook |
| 여러 layer feature 추출 | `extract_feature()`의 `outputs[6]` ~ `outputs[9]` |
| feature concatenate | `torch.cat((x6, x7, x8, x9), dim=2)` |
| TAB | `TABlock.forward()`와 `self.tablock1`, `self.tablock2` |
| SSTB | `self.swintransformer1`, `self.swintransformer2` |
| Dual Branch | `self.fc_score`, `self.fc_weight` |
| 최종 quality score | `torch.sum(f * w) / torch.sum(w)` |

## 2. 입력 이미지와 patch 개수

MANIQA는 기본적으로 `224x224` 이미지를 사용합니다.

ViT patch size가 `8`이므로 한 변의 patch 수는 `224 / 8 = 28`입니다.

따라서 전체 patch token 수는 `28 * 28 = 784`입니다.

In [ ]:
img_size = 224
patch_size = 8
input_size = img_size // patch_size
num_patches = input_size * input_size

print('input_size:', input_size)
print('num_patches:', num_patches)

## 3. ViT feature extraction

`MANIQA.__init__()`에서는 `timm.create_model('vit_base_patch8_224')`로 ViT를 만듭니다.

그리고 ViT 내부의 모든 `Block`에 forward hook을 등록합니다.

이 hook 덕분에 `self.vit(x)`를 한 번 실행하면 ViT block별 출력이 `self.save_output.outputs`에 저장됩니다.

## 4. 선택된 ViT layer feature

`extract_feature()`는 ViT block output 중 6, 7, 8, 9번을 사용합니다.

각 output에서 `[:, 1:]`를 사용하는 이유는 첫 번째 token인 cls token을 제외하고 patch token만 사용하기 위해서입니다.

각 feature shape은 보통 `(B, 784, 768)`입니다.

## 5. Feature concatenate

네 개 layer feature를 channel 방향으로 붙입니다.

```python
x = torch.cat((x6, x7, x8, x9), dim=2)
```

shape은 다음처럼 바뀝니다.

`(B, 784, 768)` 네 개 -> `(B, 784, 3072)`

## 6. TAB: Transposed Attention Block

TAB는 논문의 channel dimension attention에 해당합니다.

MANIQA는 feature를 먼저 다음처럼 바꿉니다.

```python
x = rearrange(x, 'b (h w) c -> b c (h w)')
```

즉 `(B, 784, 3072)`가 `(B, 3072, 784)`가 됩니다.

여기서 `C=3072`가 attention matrix의 축이 되므로 stage1 TAB attention weight는 `(B, 3072, 3072)`입니다.

## 7. SSTB: Scale Swin Transformer Block

TAB 후에는 다시 2D feature map으로 바꿉니다.

```python
x = rearrange(x, 'b c (h w) -> b c h w')
```

stage1에서는 `conv1`로 channel을 `3072 -> 768`로 줄인 뒤 `SwinTransformer`에 넣습니다.

SSTB 내부에서는 Swin block이 window attention과 shifted window attention을 번갈아 사용해 patch 주변의 local interaction을 만듭니다.

## 8. Stage 2

Stage 2는 같은 아이디어를 한 번 더 적용합니다.

1. `(B, 768, 28, 28)`을 `(B, 768, 784)`로 바꿈
2. TAB로 channel attention 적용
3. `conv2`로 channel을 `768 -> 384`로 줄임
4. 두 번째 SSTB 통과

결과 feature는 `(B, 384, 28, 28)`입니다.

## 9. Dual Branch와 최종 score

마지막 feature를 patch별 feature로 펼칩니다.

`(B, 384, 28, 28)` -> `(B, 784, 384)`

각 patch feature는 두 branch를 통과합니다.

- `fc_score`: patch별 quality score
- `fc_weight`: patch별 importance weight

최종 score는 patch score를 weight로 가중 평균한 값입니다.

```python
_s = torch.sum(f * w) / torch.sum(w)
```

## 10. 실제 shape 확인 코드

이 셀은 이 노트북이 `MANIQA/models` 폴더에 있기 때문에 parent 경로를 `sys.path`에 추가한 뒤 실행합니다.

VSCode kernel은 `MANIQA/.venv`를 선택하세요.

In [11]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / 'models' / 'maniqa.py').exists() else cwd.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import torch
from models.maniqa import MANIQA

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
x = torch.rand(1, 3, 224, 224).to(device)

model = MANIQA(
    embed_dim=768,
    num_outputs=1,
    dim_mlp=768,
    patch_size=8,
    img_size=224,
    window_size=4,
    depths=[2, 2],
    num_heads=[4, 4],
    num_tab=2,
    scale=0.8,
    vit_pretrained=False,
).to(device)

model.eval()
with torch.no_grad():
    score = model(x, debug=True)

print('final score:', score.detach().cpu().tolist())

[MANIQA debug] input image tensor: (1, 3, 224, 224)
[MANIQA debug] patch embedding output: (1, 784, 768)
[MANIQA debug] raw ViT model output (not used for quality score): (1, 784, 768)
[MANIQA debug] saved ViT block outputs: 12
[MANIQA debug] selected ViT layer 6 feature without cls token: (1, 784, 768)
[MANIQA debug] selected ViT layer 7 feature without cls token: (1, 784, 768)
[MANIQA debug] selected ViT layer 8 feature without cls token: (1, 784, 768)
[MANIQA debug] selected ViT layer 9 feature without cls token: (1, 784, 768)
[MANIQA debug] concatenated ViT feature: (1, 784, 3072)
[MANIQA debug] stage1 TAB input: (1, 3072, 784)
[TAB stage1 block0] input feature: (1, 3072, 784)
[TAB stage1 block0] q/k/v for channel attention: (1, 3072, 784) / (1, 3072, 784) / (1, 3072, 784)
[TAB stage1 block0] channel attention weight: (1, 3072, 3072)
[TAB stage1 block0] output feature after channel weighting + residual: (1, 3072, 784)
[TAB stage1 block1] input feature: (1, 3072, 784)
[TAB stage1 bl